In [4]:
#=====================================================================
  run_casos
  Resuelve los 4 escenarios de UC (Basico, Eolico, BESS, Completo) en
  el nodo optimo detectado por run_ubicacion y exporta todos los
  resultados a OUTPUT/CASES_DATA.
=====================================================================#
using JuMP, HiGHS
import MathOptInterface
const MOI = MathOptInterface
using XLSX, CSV, DataFrames, Printf

# ---------- RUTAS PORTABLES ----------
const SCRIPT_DIR = @__DIR__
const BASE     = dirname(SCRIPT_DIR)
const IN_BESS  = joinpath(BASE, "INPUT", "BESS")
const IN_WIND  = joinpath(BASE, "INPUT", "Wind power generation forecast")
const IN_NODE  = joinpath(BASE, "INPUT", "24 NODE MODEL")
const OUT_LOC  = joinpath(BASE, "OUTPUT", "LOCATION")
const OUT_DATA = joinpath(BASE, "OUTPUT", "CASES_DATA")
mkpath(OUT_DATA)
println("BASE del proyecto: ", BASE)

# ---------- DATOS ----------
sheet = XLSX.readxlsx(joinpath(IN_NODE,"datos_generadores.xlsx"))[1]
buses_gen = Int64.(sheet["B"][2:end]); Pmax = Float64.(sheet["C"][2:end]); Pmin = Float64.(sheet["D"][2:end])
Costo = Float64.(sheet["E"][2:end]); Costofijo = Float64.(sheet["F"][2:end]); CostoArranque = Float64.(sheet["G"][2:end])
nGen = length(Pmax)
sheet_demanda = XLSX.readxlsx(joinpath(IN_NODE,"demanda_sistema.xlsx"))[1]
demanda_horaria = Float64.(sheet_demanda["B"][2:end]); T = length(demanda_horaria)
sheet_eolico = XLSX.readxlsx(joinpath(IN_WIND,"datos_eolicos.xlsx"))[1]
num_eolicos = Int(sheet_eolico["B"][2])
ρ_vec = Float64.(sheet_eolico["C"][2]); A_vec = Float64.(sheet_eolico["D"][2]); Cp_vec = Float64.(sheet_eolico["E"][2])
sheet_viento = XLSX.readxlsx(joinpath(IN_WIND,"velocidad_viento.xlsx"))[1]
velocidades_viento = Float64.(sheet_viento["B"][2:end])
function potencia_eolica_teorica(v)
    if v < 3 || v > 25
        return 0.0
    elseif v <= 11.8
        return min(0.5 * ρ_vec[1] * A_vec[1] * Cp_vec[1] * v^3 / 1000, 1500) / 1000
    else
        return 1.5
    end
end
potencia_eolica = num_eolicos .* [potencia_eolica_teorica(v) for v in velocidades_viento]
sheet_lineas = XLSX.readxlsx(joinpath(IN_NODE,"lineas_transmision.xlsx"))[1]
nLineas = count(!ismissing, sheet_lineas["A"][2:end])
desde = Int64.(sheet_lineas["A"][2:1+nLineas]); hasta = Int64.(sheet_lineas["B"][2:1+nLineas])
reactancia = Float64.(sheet_lineas["C"][2:1+nLineas]); flujo_max = Float64.(sheet_lineas["D"][2:1+nLineas])
x = Dict((desde[i], hasta[i]) => reactancia[i] for i in 1:nLineas)
Fmax = Dict((desde[i], hasta[i]) => flujo_max[i] for i in 1:nLineas)
lineas_orden = [(desde[i],hasta[i]) for i in 1:nLineas]
nBuses = maximum(vcat(desde, hasta))
sheet_bess = XLSX.readxlsx(joinpath(IN_BESS,"datos_bess.xlsx"))[1]
Emax=Float64(sheet_bess["B"][2]); pev_max=Float64(sheet_bess["C"][2])
η_c=Float64(sheet_bess["D"][2]); η_d=Float64(sheet_bess["E"][2]); ramp_pev=Float64(sheet_bess["F"][2])
SOC_ini=Float64(sheet_bess["G"][2]); SOC_min_pct=Float64(sheet_bess["H"][2]); pen_soc_low=Float64(sheet_bess["I"][2])
carga_bess=Float64(sheet_bess["J"][2]); descarga_bess=Float64(sheet_bess["K"][2]); degradacion_bess=Float64(sheet_bess["L"][2])
sheet_carga = XLSX.readxlsx(joinpath(IN_NODE,"distribucion_de_carga.xlsx"))[1]
buses_carga = Int64.(sheet_carga["A"][2:end]); porcentajes_carga = Float64.(sheet_carga["B"][2:end])
distribucion_carga = Dict(buses_carga[i] => porcentajes_carga[i] for i in 1:length(buses_carga))

# ---------- MEJOR NODO ----------
sw = CSV.read(joinpath(OUT_LOC,"OUT_sweep.csv"), DataFrame)
imin = argmin(sw.costo); BUS_BESS_OPT = sw.bus_bess[imin]; BUS_EOL_OPT = sw.bus_eolico[imin]
@printf("Nodo optimo -> BESS=%d  EOLICO=%d  costo=%.2f\n", BUS_BESS_OPT, BUS_EOL_OPT, sw.costo[imin])
CSV.write(joinpath(OUT_DATA,"gens.csv"), DataFrame(G=1:nGen, Bus=buses_gen, Pmax=Pmax, Pmin=Pmin, Costo=Costo, Cfijo=Costofijo, Carr=CostoArranque))

# ---------- MODELO PARAMETRICO ----------
function correr(case::String, use_wind::Bool, use_bess::Bool)
    bus_eolico = BUS_EOL_OPT; bus_bess = BUS_BESS_OPT
    pe = use_wind ? potencia_eolica : zeros(T)
    model = Model(HiGHS.Optimizer); set_silent(model)
    @variable(model, p[g=1:nGen, t=1:T]); @variable(model, u[g=1:nGen, t=1:T], Bin)
    @variable(model, v[g=1:nGen, t=1:T], Bin); @variable(model, θ[b=1:nBuses, t=1:T])
    if use_wind; @variable(model, 0 <= pw[t=1:T] <= potencia_eolica[t]); end
    if use_bess
        @variable(model, 0 <= charge[t=1:T] <= pev_max); @variable(model, 0 <= discharge[t=1:T] <= pev_max)
        @variable(model, 0 <= soc[t=1:T] <= Emax); @variable(model, pen_low[t=1:T] >= 0); @variable(model, z[t=1:T], Bin)
    end
    for t in 1:T, b in 1:nBuses
        gen = sum(p[g,t] for g in 1:nGen if buses_gen[g] == b; init=0.0)
        if use_wind && b == bus_eolico; gen += pw[t]; end
        if use_bess && b == bus_bess;   gen += discharge[t]; end
        demanda_b = get(distribucion_carga, b, 0.0) * demanda_horaria[t]
        if use_bess && b == bus_bess; demanda_b += charge[t]; end
        inflow = sum((θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if j == b; init=0.0)
        outflow = sum((θ[i,t] - θ[j,t]) / x[(i,j)] for (i,j) in keys(x) if i == b; init=0.0)
        @constraint(model, gen + inflow - outflow == demanda_b)
    end
    for g in 1:nGen, t in 1:T
        @constraint(model, p[g,t] <= Pmax[g] * u[g,t]); @constraint(model, p[g,t] >= Pmin[g] * u[g,t])
    end
    for g in 1:nGen
        @constraint(model, v[g,1] >= u[g,1])
        for t in 2:T; @constraint(model, v[g,t] >= u[g,t] - u[g,t-1]); end
    end
    if use_bess
        @constraint(model, soc[1] == SOC_ini / 100 * Emax + η_c * charge[1] - discharge[1]/η_d)
        for t in 2:T
            @constraint(model, soc[t] == soc[t-1] + η_c * charge[t] - discharge[t]/η_d)
            @constraint(model, charge[t] - charge[t-1] <= ramp_pev); @constraint(model, charge[t-1] - charge[t] <= ramp_pev)
            @constraint(model, discharge[t] - discharge[t-1] <= ramp_pev); @constraint(model, discharge[t-1] - discharge[t] <= ramp_pev)
        end
        for t in 1:T; @constraint(model, pen_low[t] >= SOC_min_pct/100 * Emax - soc[t]); end
        for t in 1:T
            @constraint(model, charge[t] <= pev_max * (1 - z[t])); @constraint(model, discharge[t] <= pev_max * z[t])
        end
        @constraint(model, soc[T] >= SOC_ini / 100 * Emax)
    end
    for (i,j) in keys(x), t in 1:T
        fij = (θ[i,t] - θ[j,t]) / x[(i,j)]
        @constraint(model, fij <= Fmax[(i,j)]); @constraint(model, fij >= -Fmax[(i,j)])
    end
    for t in 1:T; @constraint(model, θ[1,t] == 0); end
    obj = @expression(model, sum(Costo[g]*p[g,t] + Costofijo[g]*u[g,t] + CostoArranque[g]*v[g,t] for g=1:nGen, t=1:T))
    if use_wind; obj += sum((potencia_eolica[t] - pw[t]) * 100 for t=1:T); end
    if use_bess
        obj += sum(pen_low[t]*pen_soc_low for t=1:T)
        obj += sum(carga_bess * charge[t] + descarga_bess * discharge[t] for t in 1:T)
        obj += sum(degradacion_bess * (charge[t] + discharge[t]) for t in 1:T)
    end
    @objective(model, Min, obj)
    tic = time(); optimize!(model); toc = time(); st = termination_status(model)
    @printf("[%s] status=%s  obj=%.2f  t=%.2fs\n", case, string(st), objective_value(model), toc-tic)

    pwv  = use_wind ? [value(pw[t]) for t in 1:T] : zeros(T)
    chv  = use_bess ? [value(charge[t]) for t in 1:T] : zeros(T)
    disv = use_bess ? [value(discharge[t]) for t in 1:T] : zeros(T)
    socv = use_bess ? [value(soc[t]) for t in 1:T] : zeros(T)
    penv = use_bess ? [value(pen_low[t]) for t in 1:T] : zeros(T)
    capcom = [sum(Pmax[g]*round(Int,value(u[g,t])) for g in 1:nGen) for t in 1:T]

    dfg = DataFrame(Hora=1:T, Demanda=demanda_horaria)
    for g in 1:nGen; dfg[!, "p$g"] = [value(p[g,t]) for t in 1:T]; end
    for g in 1:nGen; dfg[!, "u$g"] = [round(Int, value(u[g,t])) for t in 1:T]; end
    dfg[!, "Cap_comprometida"] = capcom
    dfg[!, "Eolica_disp"] = pe; dfg[!, "Eolica_uso"] = pwv; dfg[!, "Vertimiento"] = pe .- pwv
    dfg[!, "Charge"] = chv; dfg[!, "Discharge"] = disv; dfg[!, "BESS_net"] = disv .- chv
    dfg[!, "SOC_MWh"] = socv; dfg[!, "SOC_pct"] = 100 .* socv ./ Emax
    CSV.write(joinpath(OUT_DATA,"$(case)_gen.csv"), dfg)

    dff = DataFrame(Hora=1:T)
    for (i,j) in lineas_orden; dff[!, "L_$(i)_$(j)"] = [(value(θ[i,t])-value(θ[j,t]))/x[(i,j)] for t in 1:T]; end
    CSV.write(joinpath(OUT_DATA,"$(case)_flujos.csv"), dff)

    dfm = DataFrame(Hora=Int[], Linea=String[], Flujo=Float64[], Limite=Float64[], Carga_pct=Float64[])
    for t in 1:T
        best=("",0.0,0.0,0.0); bestload=-1.0
        for (i,j) in lineas_orden
            f = (value(θ[i,t])-value(θ[j,t]))/x[(i,j)]; lim = Fmax[(i,j)]; load = 100*abs(f)/lim
            if load > bestload; bestload=load; best=("$i-$j", f, lim, load); end
        end
        push!(dfm, (t, best[1], round(best[2],digits=2), best[3], round(best[4],digits=2)))
    end
    CSV.write(joinpath(OUT_DATA,"$(case)_flujomax.csv"), dfm)

    Cgen  = sum(Costo[g]*value(p[g,t]) for g=1:nGen, t=1:T)
    Cfijo = sum(Costofijo[g]*value(u[g,t]) for g=1:nGen, t=1:T)
    Carr  = sum(CostoArranque[g]*value(v[g,t]) for g=1:nGen, t=1:T)
    Cvert = use_wind ? sum((pe[t]-pwv[t])*100 for t=1:T) : 0.0
    CSOC  = use_bess ? sum(penv[t]*pen_soc_low for t=1:T) : 0.0
    CBESS = use_bess ? (sum(carga_bess*chv[t] + descarga_bess*disv[t] for t=1:T) + sum(degradacion_bess*(chv[t]+disv[t]) for t=1:T)) : 0.0
    Total = objective_value(model)
    CSV.write(joinpath(OUT_DATA,"$(case)_obj.csv"),
        DataFrame(Componente=["Cgen","Cfijo","Carr","Cvert","CSOC","CBESS","Total"],
                  Valor=[Cgen,Cfijo,Carr,Cvert,CSOC,CBESS,Total],
                  Aplica=[true,true,true,use_wind,use_bess,use_bess,true]))

    nbin = count(is_binary, all_variables(model)); ntot = num_variables(model)
    ncon = num_constraints(model; count_variable_in_set_constraints=false)
    gap = try relative_gap(model) catch; NaN end
    CSV.write(joinpath(OUT_DATA,"$(case)_dim.csv"),
        DataFrame(Metrica=["Var_continuas","Var_binarias","Var_totales","Restricciones","Estado","Gap_pct","Tiempo_s","Costo_total"],
                  Valor=[string(ntot-nbin), string(nbin), string(ntot), string(ncon),
                         string(st), @sprintf("%.4f",100*gap), @sprintf("%.3f",toc-tic), @sprintf("%.2f",Total)]))
    return Total
end

c1 = correr("Basic",  false, false); c2 = correr("Eolico", true,  false)
c3 = correr("BESS",   false, true);  c4 = correr("Final",  true,  true)
CSV.write(joinpath(OUT_DATA,"resumen_casos.csv"),
    DataFrame(Caso=["Basico","Eolico","BESS","Completo"], Costo=[c1,c2,c3,c4],
              Bus_BESS=[0,0,BUS_BESS_OPT,BUS_BESS_OPT], Bus_Eol=[0,BUS_EOL_OPT,0,BUS_EOL_OPT]))
println("OK casos -> ", OUT_DATA)


BASE del proyecto: C:\TESIS_UC\ENTREGABLES\UC MODEL
Nodo optimo -> BESS=1  EOLICO=1  costo=429870.38
[Basic] status=OPTIMAL  obj=434096.02  t=0.49s
[Eolico] status=OPTIMAL  obj=429992.40  t=0.45s
[BESS] status=OPTIMAL  obj=433973.99  t=0.58s
[Final] status=OPTIMAL  obj=429870.38  t=0.85s
OK casos -> C:\TESIS_UC\ENTREGABLES\UC MODEL\OUTPUT\CASES_DATA
